In [2]:
!pip install python-docx

In [2]:
from docx import Document
from docx.shared import Pt, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH, WD_TAB_ALIGNMENT
from docx.opc.constants import RELATIONSHIP_TYPE as RT
import docx

def add_hyperlink(paragraph, text, url):
    """Añade un hipervínculo funcional al texto."""
    part = paragraph.part
    r_id = part.relate_to(url, RT.HYPERLINK, is_external=True)

    hyperlink = docx.oxml.shared.OxmlElement('w:hyperlink')
    hyperlink.set(docx.oxml.shared.qn('r:id'), r_id)

    new_run = docx.oxml.shared.OxmlElement('w:r')
    rPr = docx.oxml.shared.OxmlElement('w:rPr')

    # Estilo visual del link (azul estándar)
    c = docx.oxml.shared.OxmlElement('w:color')
    c.set(docx.oxml.shared.qn('w:val'), '0000FF') 
    rPr.append(c)

    new_run.append(rPr)
    new_run.text = text
    hyperlink.append(new_run)
    paragraph._p.append(hyperlink)

def add_bottom_border(paragraph):
    """Línea negra fina y elegante debajo de los títulos de sección."""
    p = paragraph._p
    pPr = p.get_or_add_pPr()
    pbdr = OxmlElement('w:pBdr')
    bottom = OxmlElement('w:bottom')
    bottom.set(qn('w:val'), 'single')
    bottom.set(qn('w:sz'), '6')       
    bottom.set(qn('w:space'), '4')   
    bottom.set(qn('w:color'), '000000')
    pbdr.append(bottom)
    pPr.append(pbdr)

# --- CONFIGURACIÓN DE PÁGINA ---
doc = Document()

MARGIN_SIZE = 1.6
sections = doc.sections
for section in sections:
    section.top_margin = Cm(MARGIN_SIZE)
    section.bottom_margin = Cm(MARGIN_SIZE)
    section.left_margin = Cm(MARGIN_SIZE)
    section.right_margin = Cm(MARGIN_SIZE)

TAB_POS = Cm(17.6)

# Fuente Base: Times New Roman, 11 pt
style = doc.styles['Normal']
font = style.font
font.name = 'Times New Roman'
font.size = Pt(11)

# --- ENCABEZADO ---
header_p = doc.add_paragraph()
header_p.alignment = WD_ALIGN_PARAGRAPH.CENTER
name_run = header_p.add_run('MERCEDES GONZÁLEZ SÁNCHEZ-GRANDE')
name_run.bold = True
name_run.font.size = Pt(16)

contact_p = doc.add_paragraph()
contact_p.alignment = WD_ALIGN_PARAGRAPH.CENTER
contact_p.paragraph_format.space_after = Pt(12)

# Datos de contacto + Hipervínculo
contact_p.add_run('Madrid, España | +34 XXX XXX XXX | mxxxxxx@gmail.com\n')
add_hyperlink(contact_p, '[LinkedIn]', 'https://www.linkedin.com/in/mercedesgonzalezsg/')
contact_p.add_run(' | ')
add_hyperlink(contact_p,'[Portfolio]', 'https://github.com/mergsg')

# --- FUNCIONES AUXILIARES ---

def add_section_header(title):
    p = doc.add_paragraph()
    p.paragraph_format.space_before = Pt(10)
    p.paragraph_format.space_after = Pt(4)
    run = p.add_run(title.upper())
    run.bold = True
    run.font.size = Pt(11)
    add_bottom_border(p)

def add_entry(company, role, date, bullets=None):
    """Función unificada para Experiencia y Educación"""
    # Línea 1: Empresa | Rol ...... Fecha
    p = doc.add_paragraph()
    p.paragraph_format.space_before = Pt(6)
    p.paragraph_format.space_after = Pt(2)
    p.paragraph_format.tab_stops.add_tab_stop(TAB_POS, WD_TAB_ALIGNMENT.RIGHT)
    
    # Empresa
    r_comp = p.add_run(company)
    r_comp.bold = True
    
    p.add_run(" | ")
    
    # Rol
    r_role = p.add_run(role)
    r_role.italic = True 
    
    # Fecha
    r_date = p.add_run(f"\t{date}")
    r_date.bold = True 

    # Bullets Justificados
    if bullets:
        for bullet in bullets:
            p_bull = doc.add_paragraph(style='List Bullet')
            p_bull.paragraph_format.space_after = Pt(2)
            p_bull.paragraph_format.line_spacing = 1.05 
            p_bull.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
            p_bull.add_run(bullet)

# --- CONTENIDO ---

# 1. PERFIL
add_section_header("PERFIL PROFESIONAL")
perfil_texto = (
    "Analista de Datos con una ventaja competitiva: visión de negocio. "
    "Tras 10 años optimizando los resultados de comunicación en cuentas globales, "
    "he pivotado hacia la analítica técnica avanzada. Actualmente combino mi formación técnica en "
    "Python y SQL con la capacidad de comunicar hallazgos complejos a stakeholders y equipos de negocio."
)
p_prof = doc.add_paragraph(perfil_texto)
p_prof.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
p_prof.paragraph_format.line_spacing = 1.1

# 2. SKILLS
add_section_header("HABILIDADES TÉCNICAS")
skills = [
    ("Análisis & Programación:", " Python (Pandas y NumPy para limpieza/EDA), SQL (Consultas complejas), Excel Avanzado."),
    ("Visualización & BI:", " Tableau (Nivel Instructor), Power BI, Diseño de Dashboards de Negocio."),
    ("Automatización:", " n8n, Zapier, Google Analytics 4 (GA4).")
]
for title, desc in skills:
    p = doc.add_paragraph(style='List Bullet')
    p.paragraph_format.space_after = Pt(2)
    p.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    r = p.add_run(title)
    r.bold = True
    p.add_run(desc)

# 3. EDUCACIÓN
add_section_header("EDUCACIÓN")

add_entry(
    "ISDI", 
    "Master in Data Analytics & Artificial Intelligence", 
    "Mayo 2025 – Actualidad",
    [
        "Stack Tecnológico: Formación intensiva en ciclo de vida del dato: Ingesta (SQL), Procesamiento (Python), Visualización y Modelado (Machine Learning)."
    ]
)

add_entry(
    "UNIVERSIDAD COMPLUTENSE", 
    "Licenciatura en Periodismo y Máster en Comunicación y Marketing", 
    "2009 – 2015",
    [
        "Especialización: Comunicación y marketing en el sector Moda y Lujo."
    ]
)

# 4. EXPERIENCIA
add_section_header("EXPERIENCIA PROFESIONAL")

add_entry(
    "ISDI", 
    "Tableau Technical Support (Colaboración Académica)", 
    "Dic 2025", 
    [
        "Capacidad Técnica: Seleccionada para impartir soporte técnico intensivo en Tableau a alumnos del Máster de Digital Marketing.",
        "Resolución de Problemas: Apoyo en limpieza de datos, creación de campos calculados y optimización de visualizaciones."
    ]
)

add_entry(
    "CANELA", 
    "Senior Account Manager", 
    "Feb 2022 – Sep 2025", 
    [
        "Transformación Digital: Lideré la transición de los reports de cliente hacia una cultura data-driven, ayudando a diseñar dashboards en Tableau que redujeron la carga manual un 30%.",
        "Impacto en Negocio: Monitorización de KPIs de crecimiento para clientes globales (Dyson, Satisfyer, Pepsi, entre otros), fundamentando estrategias que lograron un +20% YoY.",
        "Gestión de equipos: Supervisión de tareas y resultados de un equipo de 6 personas."
    ]
)

add_entry(
    "IDÓNEA", 
    "Senior Account Executive", 
    "Mar 2019 – Ene 2022", 
    [
        "Análisis de ROI: Elaboración de informes de rendimiento para cuentas clave (L'Oréal), correlacionando métricas de cobertura y engagement para justificar la inversión.",
        "Estrategia basada en Datos: Análisis de desviaciones en KPIs mensuales para corregir y optimizar las tácticas de campaña en tiempo real."
    ]
)

add_entry(
    "RÉPLICA", 
    "Showroom & Account Executive", 
    "Mar 2016 – Ene 2022", 
    [
        "Eficiencia Operativa: Digitalización del sistema de control de inventario (+1.000 referencias), logrando reducir las pérdidas de stock y mejorar la eficiencia un 30%."
    ]
)


# Guardar
file_name = 'GonzálezSánchezGrandeMercedes_CV.docx'
doc.save(file_name)
print(f"CV generado: {file_name}")

CV generado: GonzálezSánchezGrandeMercedes_CV.docx
